# 五模型推理数据完整性复核

源数据：只读挂载的 `/mnt/usb`。独立基线：项目冻结的 570 案例 Weather Hub handoff。
检查范围为全部案例、文件大小和 NetCDF 结构，以及每个变量、每个预报时次的 2×2 网格抽样。
高空场每个时次抽样 200/500/850 hPa 和一个轮换层，首末时次覆盖全部 13 层；抽样位置随案例和时次轮换。
这不是全量数值扫描，也没有原始输出文件的 SHA-256 基线。核心检查实现在 `scripts/audit_inference_data.py`；本 notebook 读取逐文件证据并复核结论。
前几节保留首次审计快照，文件大小复核读取当前硬盘；末尾记录重新下载后的 Pangu 复查。

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

assert Path(sys.prefix).name == 'ai-weather-eval', '请选择 ai-weather-eval Conda 环境'
ROOT = Path.cwd()
if not (ROOT / 'scripts/audit_inference_data.py').exists():
    ROOT = ROOT.parent
EVIDENCE = ROOT / 'outputs/usb_integrity_2026-10-20'
audit = json.loads((EVIDENCE / 'summary.json').read_text())
assert audit['phase'] == 'complete', '完整检查尚未结束'
print(audit['started_utc'], audit['finished_utc'])
print('基线 SHA-256:', audit['baseline_sha256'])

In [ ]:
rows = []
for model, item in audit['models'].items():
    rows.append({
        'model': model, 'expected_cases': item['expected_cases'],
        'present_cases': item['present_cases'], 'netcdf_files': item['netcdf_files'],
        'structure_failures': item['structure']['failed_files'],
        'sample_failures': item['sample']['failed_files'],
        'sampled_values': item['sample']['sample_values'],
        'sampled_file_leads': item['sample']['sampled_leads'],
        'logical_bytes': item['netcdf_bytes'],
    })
pd.DataFrame(rows).set_index('model')

In [ ]:
details = {phase: [json.loads(line) for line in (EVIDENCE / f'{phase}.jsonl').read_text().splitlines()]
           for phase in ['structure', 'sample']}
failures = [dict(phase=phase, model=row['model'], case_id=row['case_id'],
                 path=row['path'], errors='; '.join(row['errors']))
            for phase, records in details.items() for row in records if row['errors']]
pd.DataFrame(failures)

In [ ]:
size_failures = []
for row in details['structure']:
    if not row['errors']:
        continue
    path = Path(row['path'])
    manifest_path = path.parent / 'manifest.json'
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        expected = manifest['output_sizes'][path.stem]
        actual = path.stat().st_size
        size_failures.append(dict(case_id=row['case_id'], file=path.name,
                                  expected_bytes=expected, actual_bytes=actual,
                                  missing_bytes=expected-actual))
pd.DataFrame(size_failures)

In [ ]:
metadata = json.loads((EVIDENCE / 'metadata_readability.json').read_text())
print('JSON:', metadata['checked_json'], 'CSV:', metadata['checked_csv'])
print('解析错误:', metadata['errors'])
assert not metadata['errors']

## 重新下载后的 Pangu 复查

下列结果使用独立输出目录保留首次审计证据。全部 Pangu 文件重复进行结构和逐时次抽样检查；
之前截断的两个 upper.nc 额外逐一读取所有变量、时次、气压层和完整网格，并计算本次 SHA-256。
全量读取范围仅为这两个文件。执行记录和完整读取代码保存在对应 outputs 目录。

In [ ]:
RECHECK = ROOT / 'outputs/pangu_integrity_recheck_2026-10-20'
pangu_recheck = json.loads((RECHECK / 'summary.json').read_text())
assert pangu_recheck['phase'] == 'complete'
assert set(pangu_recheck['models']) == {'pangu'}
print('Pangu 复查通过:', pangu_recheck['passed'])
print('案例/文件:', pangu_recheck['models']['pangu']['present_cases'],
      pangu_recheck['models']['pangu']['netcdf_files'])
repaired = json.loads((RECHECK / 'repaired_full_read.json').read_text())
print('两个修复文件全量读取通过:', repaired['passed'])
pd.DataFrame([{k: row[k] for k in ['case_id', 'expected_bytes', 'actual_bytes',
                                  'checked_values', 'passed', 'sha256']}
              for row in repaired['files']])

完整重跑（使用新的输出目录保留本次证据）：

```bash
conda run --name ai-weather-eval --no-capture-output python scripts/audit_inference_data.py \
  --root /mnt/usb --output outputs/usb_integrity_recheck --workers 4
```
单模型复查可增加 `--models pangu`。

`*-inventory.csv` 保存盘内全部普通文件的路径和大小；`summary.json` 保存案例与清单交叉检查；
`structure.jsonl` 和 `sample.jsonl` 保存每个 NetCDF 文件的检查结果。